### Data Ingestion


In [4]:
###Document Structure

from langchain_core.documents import Document

In [5]:
doc = Document(
    page_content="sample content for rag",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author": 'Hari',
        "date": "2026-09-15"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Hari', 'date': '2026-09-15'}, page_content='sample content for rag')

In [6]:
### create simple file

import os
os.makedirs("../data/text_files",exist_ok=True)

In [7]:
sample_texts={
    "../data/text_files/python_101.txt": """Python is a popular, high-level programming language designed to have a clean, easy-to-read syntax that resembles everyday English.Key CharacteristicsEasy to read: Uses whitespace and indentation instead of heavy brackets or punctuation, making code simple to maintain.Interpreted: Executes code line by line, which helps you spot and fix errors quickly during development.Free and open-source: Available to download, use, and modify for any purpose via the Python Official Website.Cross-platform: Runs smoothly on Windows, macOS, Linux, and other major operating systems.Extensive libraries: Offers thousands of pre-built modules and packages for tasks ranging from math to networking.Common UsesArtificial Intelligence and Machine Learning: Powers tools used for training smart algorithms, data modeling, and neural networks.Data Science: Analyzes and visualizes large volumes of data using specialized toolkits.Web Development: Builds robust backend servers and web applications using frameworks like Django and Flask.Automation: Automates repetitive daily tasks, file management, and system administration through simple scripts."""
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write(content)

print("Sample file created!!")

Sample file created!!


In [8]:
### Text loader

from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_101.txt", encoding="utf-8")
document = loader.load()
print(document)

C:\Users\haryo\AppData\Local\Temp\ipykernel_41796\2636267989.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\coding\python\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_101.txt'}, page_content='Python is a popular, high-level programming language designed to have a clean, easy-to-read syntax that resembles everyday English.Key CharacteristicsEasy to read: Uses whitespace and indentation instead of heavy brackets or punctuation, making code simple to maintain.Interpreted: Executes code line by line, which helps you spot and fix errors quickly during development.Free and open-source: Available to download, use, and modify for any purpose via the Python Official Website.Cross-platform: Runs smoothly on Windows, macOS, Linux, and other major operating systems.Extensive libraries: Offers thousands of pre-built modules and packages for tasks ranging from math to networking.Common UsesArtificial Intelligence and Machine Learning: Powers tools used for training smart algorithms, data modeling, and neural networks.Data Science: Analyzes and visualizes large volumes of data using specialized toolkits.Web 

In [9]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=False,
    loader_kwargs={"encoding":'utf-8'}
)
d_load =dir_loader.load()
d_load

[Document(metadata={'source': '..\\data\\text_files\\python_101.txt'}, page_content='Python is a popular, high-level programming language designed to have a clean, easy-to-read syntax that resembles everyday English.Key CharacteristicsEasy to read: Uses whitespace and indentation instead of heavy brackets or punctuation, making code simple to maintain.Interpreted: Executes code line by line, which helps you spot and fix errors quickly during development.Free and open-source: Available to download, use, and modify for any purpose via the Python Official Website.Cross-platform: Runs smoothly on Windows, macOS, Linux, and other major operating systems.Extensive libraries: Offers thousands of pre-built modules and packages for tasks ranging from math to networking.Common UsesArtificial Intelligence and Machine Learning: Powers tools used for training smart algorithms, data modeling, and neural networks.Data Science: Analyzes and visualizes large volumes of data using specialized toolkits.W

In [10]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False,
)
d_load =dir_loader.load()
d_load

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'source': '..\\data\\pdf\\2506.18027v3.pdf', 'file_path': '..\\data\\pdf\\2506.18027v3.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': 'PDF Retrieval Augmented Question Answering', 'author': 'Thi Thu Uyen Hoang; Meenakshi Rajendran; Kun Zhang; Yuhan Wu; Viet Anh Nguyen', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='PDF Retrieval Augmented Question Answering\nThi Thu Uyen Hoang\nMeenakshi Rajendran\nKun Zhang\nYuhan Wu\nViet Anh Nguyen\nSaarland University\n{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de\nAbstract\nThis paper presents an advancement in Question-Answering (QA) systems using\na Retrieval Augmented Generation (RAG) framework to enhance information\nextraction from PDF files. Recognizing the richness and diversity of data within\nPDFs—including text,

### Splitting to Chunks


In [11]:

from typing import List

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


class DocumentChunker:
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        if chunk_overlap >= chunk_size:
            raise ValueError("chunk_overlap must be smaller than chunk_size")

        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
            add_start_index=True,
        )

    def split(self, documents: List[Document]) -> List[Document]:
        if not documents:
            return []

        chunks = self.splitter.split_documents(documents)

        for index, chunk in enumerate(chunks):
            chunk.metadata["chunk_index"] = index

        return chunks

chunker = DocumentChunker(chunk_size=500, chunk_overlap=50)

chunks = chunker.split(d_load)

print(f"Documents: {len(d_load)}")
print(f"Chunks: {len(chunks)}")
print(chunks[0])

Documents: 16
Chunks: 107
page_content='PDF Retrieval Augmented Question Answering
Thi Thu Uyen Hoang
Meenakshi Rajendran
Kun Zhang
Yuhan Wu
Viet Anh Nguyen
Saarland University
{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de
Abstract
This paper presents an advancement in Question-Answering (QA) systems using
a Retrieval Augmented Generation (RAG) framework to enhance information
extraction from PDF files. Recognizing the richness and diversity of data within' metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'source': '..\\data\\pdf\\2506.18027v3.pdf', 'file_path': '..\\data\\pdf\\2506.18027v3.pdf', 'total_pages': 14, 'format': 'PDF 1.7', 'title': 'PDF Retrieval Augmented Question Answering', 'author': 'Thi Thu Uyen Hoang; Meenakshi Rajendran; Kun Zhang; Yuhan Wu; Viet Anh Nguyen', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'start_index': 0

### Embedding and VectorStoreDB


In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    """Handles document embedding generation using Sentence Transformer"""

    def __init__(self, model_name:str='all-MiniLM-L6-v2'):
        """
        Initalize embedding manager
        
        Args: 
        model_name : HugginFace model name for embedding
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load Transformer Model"""
        try:
            print(f"Loaded model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded, dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error in loading : {e}")
            raise

    def generate_embedddings(self, texts: List[str])->np.ndarray:
        """
        Generate Embeddings for list of strings

        Args: List of strs

        Returns: numpy array with shape(len(texts), embedding_dimension)
        """

        if not self.model:
            raise ValueError('Model not loaded')
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Embedded {embeddings.shape} for {texts}")
        return embeddings


### initialize embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loaded model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 351.97it/s]


Model loaded, dimensions: 384


### Vector Store


In [14]:
class VectorStore:
    """ Manages document embeddings in a chroma DB Vector Store"""

    def __init__(self, collection_name:str="pdf_docs", persist_directory:str="../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None
        self._initialize_store()

    def _initialize_store(self):
        """Initailze chroma db client"""

        try:
            ## Create dir and initailize client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            ## create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"Test Desc for RAG Pipeline", "hnsw:space": "cosine"}
            )

            print(f"Collection created {self.collection_name} with count as {self.collection.count}")

        except Exception as e:
            print(f"Error occured: {e}")
            raise

    def add_document(self, docs:List[any], embeddings: np.ndarray):
        """Add docs and embeddding to Vector store"""

        if len(docs) != len(embeddings):
            raise ValueError('Does not match')

        print(f"len of docs to add in Vector store is {len(docs)}")

        ids=[]
        metadata=[]
        docs_texts=[]
        embedding_list=[]

        for i, (doc, embedd) in enumerate(zip(docs,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # metadata prep
            metdta = dict(doc.metadata)
            metdta['doc_index']=i
            metdta['content_length']=len(doc.page_content)
            metadata.append(metdta)

            # doc content
            docs_texts.append(doc.page_content)

            # embedding
            embedding_list.append(embedd.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadata,
                documents=docs_texts
            )
            print(f'Collection count {self.collection}')
        except Exception as e:
            print(f"Error {e}")
            raise


### Split and Embed


In [15]:
chunk_texts = [chunk.page_content for chunk in chunks]

vector_store = VectorStore()
embeddings = embedding_manager.generate_embedddings(chunk_texts)

vector_store.add_document(
    docs=chunks,
    embeddings=embeddings,
)

Collection created pdf_docs with count as <bound method Collection.count of Collection(name=pdf_docs)>


Batches: 100%|██████████| 4/4 [00:07<00:00,  1.82s/it]


Embedded (107, 384) for ['PDF Retrieval Augmented Question Answering\nThi Thu Uyen Hoang\nMeenakshi Rajendran\nKun Zhang\nYuhan Wu\nViet Anh Nguyen\nSaarland University\n{thho00003, mera00002, kuzh00001, yuwu00001, ving00001}@stud.uni-saarland.de\nAbstract\nThis paper presents an advancement in Question-Answering (QA) systems using\na Retrieval Augmented Generation (RAG) framework to enhance information\nextraction from PDF files. Recognizing the richness and diversity of data within', 'PDFs—including text, images, vector diagrams, graphs, and tables—poses unique\nchallenges for existing QA systems primarily designed for textual content. We seek\nto develop a comprehensive RAG-based QA system that will effectively address\ncomplex multi-modal questions, where several data types are put together in the\nquestion. This is mainly achieved by refining approaches toward processing and\nintegrating non-textual elements in PDFs into the RAG framework to derive precise', 'and relevant answers,

### RAG Retreival Pipeline


In [16]:
class RAGRetriever:
    """Retrieve query based from vector store"""

    def __init__(self, vectorstore: VectorStore, embedding_manager:EmbeddingManager):
        self.vectorstore= vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int=5, score_threshold:float=0.0) -> List[Dict[str,Any]]:
        """top_k: no of top results to be returned"""

        print(f"Retrieval query: {query}")

        query_embedding = self.embedding_manager.generate_embedddings([query])[0]

        try:
            results = self.vectorstore.collection.query(
                query_embeddings=query_embedding,
                n_results=top_k
            )

            retrieved_docs =[]

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results["ids"][0]

                for i, (document, metadata, distance, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                    # Convert distnace to similarity score
                    similarity_score = 1- distance

                    if(similarity_score > score_threshold):
                        retrieved_docs.append({
                            "id":doc_id,
                            "content":document,
                            "metadata":metadata,
                            "similarity_score":similarity_score,
                            "distance":distance,
                            "rank":i+1,
                        })
                print(f"Retrieved docs: {retrieved_docs}")
            else:
                print(f"Failed to retrieve")

            return retrieved_docs
            

        except Exception as e:
            print(f'Error as {e}')
            raise

rag_ret = RAGRetriever(vectorstore=vector_store, embedding_manager=embedding_manager)





### Integration of Vector Pipeline with LLM Output

In [24]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv

groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(groq_api_key=groq_api_key, model_name="openai/gpt-oss-20b", temperature=0.1, max_tokens=1024)

### Simple RAG function

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query=query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No Relevant context found"

    ### include llm for answer

    prompt = f"""Use the following context to answer the question precisely
    
    Context:
    {context}

    Question: {query}

    Answer:

    """

    response = llm.invoke([prompt])

    return response.content

answer = rag_simple('What is Embedding and Storage?', rag_ret, llm=llm)
print(answer)

Retrieval query: What is Embedding and Storage?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.56it/s]


Embedded (1, 384) for ['What is Embedding and Storage?']
Retrieved docs: [{'id': 'doc_4b540ffc_29', 'content': 'the storage space leading to easier data access, understanding and manipulation.\n3.2.2\nEmbedding and storage\nThe first step involves breaking down the processed document ˆ\nDi\nmarkdown into smaller chunks\n{c1, c2, ...cm}. Data consisting of text, image captions and dictionary-formatted tables into separate\nsegments. Each chunk Cj is embedded into a high-dimensional vector space using an embedding\nmodel fembed:\nfembed(cj) →vj ∈Rd\n(3)', 'metadata': {'creationDate': '', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'start_index': 1910, 'trapped': '', 'content_length': 435, 'producer': 'pikepdf 8.15.1', 'file_path': '..\\data\\pdf\\2506.18027v3.pdf', 'page': 4, 'chunk_index': 29, 'author': 'Thi Thu Uyen Hoang; Meenakshi Rajendran; Kun Zhang; Yuhan Wu; Viet Anh Nguyen', 'format': 'PDF 1.7', 'subject': '', 'source': '..\\data\\pdf\\2506.18027v3.pdf', 'total_pages': 14, 'key